In [ ]:
import kaggle_benchmarks as kbench

# ================================================================================
# CONTRADICTORY INSTRUCTIONS GAME
# Two instances of the same LLM receive slightly different task specifications.
# Neither knows the other has different instructions. They must discover the
# contradictions through conversation, resolve them, and produce a joint output.
# Scored by a separate judge LLM (kbench.judge_llm) in its own isolated chat.
# ================================================================================

# --------------------------------------------------------------------------------
# 1. SCENARIO SUITE — Fixed, Hardcoded, Identical for All Models
# --------------------------------------------------------------------------------

SCENARIO_SUITE = [
    {
        "name": "Scenario 1: Conference Room Booking",
        "tier": "Easy",
        "turn_limit": 6,
        "shared_task": (
            "You and your partner need to schedule a 3-hour team meeting "
            "for next week. Work together to agree on all the details and "
            "produce a final meeting plan."
        ),
        "alpha_spec": [
            "The meeting MUST be on Tuesday.",
            "Maximum 8 attendees.",
            "The room must have a projector.",
            "Catering budget is $200.",
        ],
        "beta_spec": [
            "The meeting MUST be on Thursday.",
            "Maximum 8 attendees.",
            "The room must have a projector.",
            "Catering budget is $200.",
        ],
        "contradictions": [
            {
                "alpha": "Tuesday",
                "beta": "Thursday",
                "description": "Meeting day: Alpha says Tuesday, Beta says Thursday",
            },
        ],
        "common": [
            "Maximum 8 attendees",
            "Room must have projector",
            "Catering budget $200",
        ],
        "judge_criteria": {
            "detection": [
                "The conversation explicitly identifies that one player was told Tuesday and the other was told Thursday for the meeting day.",
            ],
            "resolution": [
                "The players proposed a specific resolution to the Tuesday vs Thursday conflict (e.g., picking one day, suggesting a compromise like Wednesday, or scheduling two meetings).",
            ],
            "cooperation": [
                "Both players shared their own specific requirements with each other (not just agreeing generically, but stating what they were told).",
                "The players asked each other clarifying questions about their requirements.",
            ],
            "deliverable": [
                "Both final deliverables agree on the SAME meeting day (they don't contradict each other).",
                "Both final deliverables include the common requirements: 8 attendees, projector, $200 budget.",
            ],
        },
    },
    {
        "name": "Scenario 2: Product Naming",
        "tier": "Easy",
        "turn_limit": 6,
        "shared_task": (
            "You and your partner need to name a new eco-friendly water bottle "
            "product. Agree on a final product name and a short tagline."
        ),
        "alpha_spec": [
            "The product name MUST start with 'Aqua'.",
            "The name must be at most 2 words.",
            "Target audience is athletes.",
            "The tagline should emphasize performance.",
        ],
        "beta_spec": [
            "The product name MUST start with 'Eco'.",
            "The name must be at most 2 words.",
            "Target audience is athletes.",
            "The tagline should emphasize performance.",
        ],
        "contradictions": [
            {
                "alpha": "Aqua",
                "beta": "Eco",
                "description": "Name prefix: Alpha says must start with 'Aqua', Beta says must start with 'Eco'",
            },
        ],
        "common": [
            "At most 2 words",
            "Target audience is athletes",
            "Tagline emphasizes performance",
        ],
        "judge_criteria": {
            "detection": [
                "The conversation explicitly identifies that one player was told the name must start with 'Aqua' and the other was told it must start with 'Eco'.",
            ],
            "resolution": [
                "The players proposed a specific resolution to the naming conflict (e.g., picking one prefix, combining them, or finding a creative alternative).",
            ],
            "cooperation": [
                "Both players shared their own naming requirements with each other.",
                "The players discussed the naming constraint openly rather than one simply overriding the other.",
            ],
            "deliverable": [
                "Both final deliverables propose the SAME product name (not different names).",
                "Both final deliverables include a tagline that emphasizes performance for athletes.",
            ],
        },
    },
    {
        "name": "Scenario 3: Recipe Creation",
        "tier": "Medium",
        "turn_limit": 8,
        "shared_task": (
            "You and your partner need to create a dinner recipe for 4 people. "
            "Agree on all ingredients and cooking steps, then produce the final recipe."
        ),
        "alpha_spec": [
            "The dish MUST be fully vegetarian (no meat, no fish).",
            "Cooking time must be under 30 minutes.",
            "The cuisine should be Italian.",
            "Total ingredient cost must be under $25.",
        ],
        "beta_spec": [
            "The dish MUST include chicken as a main protein.",
            "Cooking time must be under 30 minutes.",
            "The cuisine should be Italian.",
            "Total ingredient cost must be under $25.",
        ],
        "contradictions": [
            {
                "alpha": "vegetarian",
                "beta": "chicken",
                "description": "Dietary requirement: Alpha says fully vegetarian, Beta says must include chicken",
            },
        ],
        "common": [
            "Under 30 minutes cooking time",
            "Italian cuisine",
            "Under $25 total cost",
        ],
        "judge_criteria": {
            "detection": [
                "The conversation explicitly identifies that one player requires a vegetarian dish and the other requires chicken.",
            ],
            "resolution": [
                "The players proposed a specific resolution to the vegetarian vs chicken conflict (e.g., choosing one, making a dish that could optionally include chicken, or finding a middle ground).",
            ],
            "cooperation": [
                "Both players clearly stated their dietary requirement to each other.",
                "The players engaged in genuine back-and-forth discussion rather than one immediately conceding.",
            ],
            "deliverable": [
                "Both final deliverables describe the SAME dish (not two different recipes).",
                "Both final deliverables maintain the agreed common constraints: Italian, under 30 min, under $25.",
            ],
        },
    },
    {
        "name": "Scenario 4: Travel Itinerary",
        "tier": "Medium",
        "turn_limit": 8,
        "shared_task": (
            "You and your partner need to plan a 3-day weekend trip for a couple. "
            "Agree on destination, activities, and budget, then produce the final itinerary."
        ),
        "alpha_spec": [
            "The destination MUST be a beach location.",
            "Total budget is $2000.",
            "Must include a snorkeling activity.",
            "Departure is on Friday evening.",
        ],
        "beta_spec": [
            "The destination MUST be a mountain location.",
            "Total budget is $2000.",
            "Must include a hiking activity.",
            "Departure is on Friday evening.",
        ],
        "contradictions": [
            {
                "alpha": "beach",
                "beta": "mountain",
                "description": "Destination type: Alpha says beach, Beta says mountain",
            },
            {
                "alpha": "snorkeling",
                "beta": "hiking",
                "description": "Activity: Alpha says snorkeling, Beta says hiking",
            },
        ],
        "common": [
            "Budget $2000",
            "Departure Friday evening",
        ],
        "judge_criteria": {
            "detection": [
                "The conversation explicitly identifies the beach vs mountain destination conflict.",
                "The conversation explicitly identifies the snorkeling vs hiking activity conflict.",
            ],
            "resolution": [
                "The players proposed a specific resolution for the destination conflict.",
                "The players proposed a specific resolution for the activity conflict.",
            ],
            "cooperation": [
                "Both players shared their full set of requirements openly.",
                "The players worked collaboratively rather than one dominating the discussion.",
            ],
            "deliverable": [
                "Both final deliverables describe the SAME destination and activities.",
                "Both final deliverables respect the $2000 budget and Friday departure.",
            ],
        },
    },
    {
        "name": "Scenario 5: Job Posting",
        "tier": "Medium",
        "turn_limit": 8,
        "shared_task": (
            "You and your partner need to write a job posting for a software engineer. "
            "Agree on all requirements, then produce the final job description."
        ),
        "alpha_spec": [
            "The position MUST be fully remote.",
            "Minimum 3 years of experience required.",
            "Python is a required skill.",
            "Salary range is $120k-$140k.",
        ],
        "beta_spec": [
            "The position MUST be on-site in New York City.",
            "Minimum 3 years of experience required.",
            "Python is a required skill.",
            "Salary range is $120k-$140k.",
        ],
        "contradictions": [
            {
                "alpha": "remote",
                "beta": "on-site",
                "description": "Work location: Alpha says fully remote, Beta says on-site in NYC",
            },
        ],
        "common": [
            "3 years experience minimum",
            "Python required",
            "Salary $120k-$140k",
        ],
        "judge_criteria": {
            "detection": [
                "The conversation explicitly identifies that one player requires remote work and the other requires on-site in NYC.",
            ],
            "resolution": [
                "The players proposed a specific resolution to the remote vs on-site conflict (e.g., hybrid model, choosing one, adjusting salary for NYC cost of living).",
            ],
            "cooperation": [
                "Both players stated their work location requirement clearly.",
                "The players discussed trade-offs or implications of each option.",
            ],
            "deliverable": [
                "Both final deliverables agree on the SAME work arrangement.",
                "Both final deliverables include Python, 3 years experience, and a salary range.",
            ],
        },
    },
    {
        "name": "Scenario 6: Event Planning",
        "tier": "Hard",
        "turn_limit": 10,
        "shared_task": (
            "You and your partner need to plan a company holiday party for 50 people. "
            "Agree on all details, then produce the final event plan."
        ),
        "alpha_spec": [
            "The venue MUST be outdoors.",
            "The event is in December.",
            "Entertainment must be a live band.",
            "All catering must be vegan.",
            "Total budget is $5000.",
        ],
        "beta_spec": [
            "The venue MUST be indoors.",
            "The event is in December.",
            "Entertainment must be a DJ.",
            "All catering must be vegan.",
            "Total budget is $5000.",
        ],
        "contradictions": [
            {
                "alpha": "outdoors",
                "beta": "indoors",
                "description": "Venue type: Alpha says outdoors, Beta says indoors",
            },
            {
                "alpha": "live band",
                "beta": "DJ",
                "description": "Entertainment: Alpha says live band, Beta says DJ",
            },
        ],
        "common": [
            "December timing",
            "Vegan catering",
            "Budget $5000",
        ],
        "judge_criteria": {
            "detection": [
                "The conversation identifies the outdoors vs indoors venue conflict.",
                "The conversation identifies the live band vs DJ entertainment conflict.",
            ],
            "resolution": [
                "The players proposed a resolution for the venue conflict.",
                "The players proposed a resolution for the entertainment conflict.",
            ],
            "cooperation": [
                "Both players shared all of their requirements, not just the conflicting ones.",
                "The players considered practical implications (e.g., outdoor event in December weather).",
            ],
            "deliverable": [
                "Both final deliverables agree on the same venue type and entertainment choice.",
                "Both final deliverables include vegan catering, December timing, and the $5000 budget.",
            ],
        },
    },
    {
        "name": "Scenario 7: Marketing Campaign",
        "tier": "Hard",
        "turn_limit": 10,
        "shared_task": (
            "You and your partner need to design a social media campaign for a new app launch. "
            "Agree on target audience, platform, tone, and content plan."
        ),
        "alpha_spec": [
            "Target audience is ages 18-25.",
            "The campaign must be Instagram-only.",
            "The tone must be playful and irreverent.",
            "Launch day is Monday.",
            "Produce exactly 5 posts.",
        ],
        "beta_spec": [
            "Target audience is ages 35-50.",
            "The campaign must be LinkedIn-only.",
            "The tone must be professional and authoritative.",
            "Launch day is Monday.",
            "Produce exactly 5 posts.",
        ],
        "contradictions": [
            {
                "alpha": "18-25",
                "beta": "35-50",
                "description": "Target age: Alpha says 18-25, Beta says 35-50",
            },
            {
                "alpha": "Instagram",
                "beta": "LinkedIn",
                "description": "Platform: Alpha says Instagram, Beta says LinkedIn",
            },
            {
                "alpha": "playful",
                "beta": "professional",
                "description": "Tone: Alpha says playful/irreverent, Beta says professional/authoritative",
            },
        ],
        "common": [
            "Launch on Monday",
            "Exactly 5 posts",
        ],
        "judge_criteria": {
            "detection": [
                "The conversation identifies the age group conflict (18-25 vs 35-50).",
                "The conversation identifies the platform conflict (Instagram vs LinkedIn).",
                "The conversation identifies the tone conflict (playful vs professional).",
            ],
            "resolution": [
                "The players proposed resolutions that address the interconnected nature of the conflicts (age, platform, and tone are linked).",
            ],
            "cooperation": [
                "Both players shared all three of their differing requirements.",
                "The players recognized that the three contradictions are related rather than treating them in isolation.",
            ],
            "deliverable": [
                "Both final deliverables agree on the same target audience, platform, and tone.",
                "Both final deliverables include 5 posts launching on Monday.",
            ],
        },
    },
    {
        "name": "Scenario 8: Interior Design",
        "tier": "Hard",
        "turn_limit": 10,
        "shared_task": (
            "You and your partner need to design the layout and style for a small cafe. "
            "Agree on aesthetic, materials, seating, and features."
        ),
        "alpha_spec": [
            "The style MUST be minimalist.",
            "Maximum 20 seats.",
            "Materials should be natural wood throughout.",
            "No TVs or screens anywhere in the cafe.",
            "Budget is $15,000.",
        ],
        "beta_spec": [
            "The style MUST be industrial.",
            "Maximum 20 seats.",
            "Materials should be metal and exposed concrete.",
            "Must include 3 TVs for showing live sports.",
            "Budget is $15,000.",
        ],
        "contradictions": [
            {
                "alpha": "minimalist",
                "beta": "industrial",
                "description": "Style: Alpha says minimalist, Beta says industrial",
            },
            {
                "alpha": "natural wood",
                "beta": "metal and concrete",
                "description": "Materials: Alpha says natural wood, Beta says metal/concrete",
            },
            {
                "alpha": "no TVs",
                "beta": "3 TVs",
                "description": "TVs: Alpha says no TVs, Beta says 3 TVs for sports",
            },
        ],
        "common": [
            "Maximum 20 seats",
            "Budget $15,000",
        ],
        "judge_criteria": {
            "detection": [
                "The conversation identifies the minimalist vs industrial style conflict.",
                "The conversation identifies the wood vs metal/concrete materials conflict.",
                "The conversation identifies the no-TVs vs 3-TVs conflict.",
            ],
            "resolution": [
                "The players proposed resolutions that account for how style, materials, and TV policy are interrelated.",
            ],
            "cooperation": [
                "Both players shared their complete vision rather than just individual constraints.",
                "The players discussed how the conflicting elements form a coherent whole (style drives material and feature choices).",
            ],
            "deliverable": [
                "Both final deliverables describe the same cohesive cafe design.",
                "Both final deliverables respect the 20-seat max and $15,000 budget.",
            ],
        },
    },
    {
        "name": "Scenario 9: Remote Work Policy",
        "tier": "Expert",
        "turn_limit": 12,
        "shared_task": (
            "You and your partner need to draft a company remote work policy. "
            "Agree on all terms, then produce the final policy document."
        ),
        "alpha_spec": [
            "Employees can work from anywhere in the world.",
            "Core collaboration hours are 9am-12pm Eastern Time.",
            "Equipment stipend is $500 per employee.",
            "Quarterly in-person meetups are held in San Francisco.",
        ],
        "beta_spec": [
            "Employees must reside within 50 miles of a company office.",
            "Core collaboration hours are 9am-12pm in the employee's local time zone.",
            "Equipment stipend is $500 per employee.",
            "Quarterly in-person meetups are held in San Francisco.",
        ],
        "contradictions": [
            {
                "alpha": "anywhere in the world",
                "beta": "within 50 miles",
                "description": "Location: Alpha says work from anywhere globally, Beta says within 50 miles of office",
            },
            {
                "alpha": "Eastern Time",
                "beta": "local time zone",
                "description": "Core hours timezone: Alpha says Eastern Time, Beta says employee's local timezone",
            },
        ],
        "common": [
            "$500 equipment stipend",
            "Quarterly meetups in San Francisco",
        ],
        "judge_criteria": {
            "detection": [
                "The conversation identifies the global-anywhere vs within-50-miles location conflict.",
                "The conversation identifies the Eastern Time vs local timezone conflict for core hours.",
            ],
            "resolution": [
                "The players proposed a resolution for the location policy.",
                "The players proposed a resolution for the timezone policy.",
                "The players considered second-order implications (e.g., global workers attending SF meetups, timezone feasibility for collaboration).",
            ],
            "cooperation": [
                "Both players shared their full policy requirements.",
                "The players discussed practical trade-offs and real-world implications of each approach.",
            ],
            "deliverable": [
                "Both final deliverables describe the SAME location and timezone policy.",
                "Both final deliverables include the $500 stipend and SF quarterly meetups.",
                "The final policies are internally consistent (location policy and timezone policy don't contradict each other).",
            ],
        },
    },
    {
        "name": "Scenario 10: Database Architecture",
        "tier": "Expert",
        "turn_limit": 12,
        "shared_task": (
            "You and your partner need to design the database architecture for a new "
            "e-commerce platform. Agree on technology choices and constraints."
        ),
        "alpha_spec": [
            "The database MUST be PostgreSQL.",
            "Inventory updates must be real-time (strong consistency).",
            "Maximum query latency is 100ms.",
            "Deployment must be single-region.",
            "Monthly infrastructure budget is $2000.",
        ],
        "beta_spec": [
            "The database MUST be MongoDB.",
            "Eventual consistency for inventory is acceptable.",
            "Maximum query latency is 100ms.",
            "Deployment must be multi-region.",
            "Monthly infrastructure budget is $2000.",
        ],
        "contradictions": [
            {
                "alpha": "PostgreSQL",
                "beta": "MongoDB",
                "description": "Database technology: Alpha says PostgreSQL, Beta says MongoDB",
            },
            {
                "alpha": "real-time strong consistency",
                "beta": "eventual consistency",
                "description": "Consistency model: Alpha says real-time/strong, Beta says eventual",
            },
            {
                "alpha": "single-region",
                "beta": "multi-region",
                "description": "Deployment: Alpha says single-region, Beta says multi-region",
            },
        ],
        "common": [
            "100ms max query latency",
            "$2000 monthly budget",
        ],
        "judge_criteria": {
            "detection": [
                "The conversation identifies the PostgreSQL vs MongoDB conflict.",
                "The conversation identifies the strong consistency vs eventual consistency conflict.",
                "The conversation identifies the single-region vs multi-region deployment conflict.",
            ],
            "resolution": [
                "The players proposed resolutions that consider the technical interdependencies (e.g., multi-region affects consistency model, database choice affects consistency guarantees).",
            ],
            "cooperation": [
                "Both players shared their complete technical requirements.",
                "The players discussed technical trade-offs and how the three conflicts are interrelated.",
            ],
            "deliverable": [
                "Both final deliverables describe the SAME database technology, consistency model, and deployment topology.",
                "Both final deliverables respect the 100ms latency and $2000 budget constraints.",
                "The final architecture is technically coherent (the chosen database supports the chosen consistency model in the chosen deployment topology).",
            ],
        },
    },
]


# --------------------------------------------------------------------------------
# 2. SYSTEM PROMPTS
# --------------------------------------------------------------------------------
PLAYER_SYSTEM_TEMPLATE = (
    "You are collaborating with a partner to complete a task. You have been "
    "given a task description and a specification with specific requirements. "
    "Your partner has also been given the task — you should work together to "
    "produce the best possible result.\n\n"
    "=== TASK ===\n{shared_task}\n\n"
    "=== YOUR REQUIREMENTS ===\n{spec}\n\n"
    "Important:\n"
    "- Share your understanding of the requirements with your partner.\n"
    "- If you notice any disagreements or inconsistencies between your "
    "requirements and your partner's, raise them explicitly.\n"
    "- Work together to resolve any conflicts and agree on a joint plan.\n"
    "- Be specific about what your requirements say.\n\n"
    "Respond with MESSAGE: followed by your message to your partner."
)

FINAL_PROMPT_TEMPLATE = (
    "The discussion phase is over. Based on your conversation with your "
    "partner, produce your FINAL meeting plan / deliverable.\n\n"
    "Original task: {shared_task}\n\n"
    "Your original requirements:\n{spec}\n\n"
    "Conversation summary:\n{conversation}\n\n"
    "Now produce your final deliverable. Include ALL agreed-upon details. "
    "If there were contradictions that you resolved, use the resolved version.\n\n"
    "Respond with DELIVERABLE: followed by your final plan."
)


def build_player_system(shared_task, spec_list):
    spec_text = "\n".join(f"- {s}" for s in spec_list)
    return PLAYER_SYSTEM_TEMPLATE.format(shared_task=shared_task, spec=spec_text)


def build_final_prompt(shared_task, spec_list, conversation_text):
    spec_text = "\n".join(f"- {s}" for s in spec_list)
    return FINAL_PROMPT_TEMPLATE.format(
        shared_task=shared_task, spec=spec_text, conversation=conversation_text
    )


# --------------------------------------------------------------------------------
# 3. RESPONSE PARSERS
# --------------------------------------------------------------------------------
def parse_message(raw):
    try:
        if "MESSAGE:" in raw:
            return raw.split("MESSAGE:")[1].strip()
    except Exception:
        pass
    return raw.strip() if raw else ""


def parse_deliverable(raw):
    try:
        if "DELIVERABLE:" in raw:
            return raw.split("DELIVERABLE:")[1].strip()
    except Exception:
        pass
    return raw.strip() if raw else ""


# --------------------------------------------------------------------------------
# 4. JUDGE-BASED SCORING
# --------------------------------------------------------------------------------
def build_judge_text(conversation_lines, alpha_deliverable, beta_deliverable, scenario):
    convo = "\n".join(conversation_lines)
    contradictions_desc = "\n".join(
        f"  - {c['description']}" for c in scenario["contradictions"]
    )
    return (
        f"=== TASK ===\n{scenario['shared_task']}\n\n"
        f"=== KNOWN CONTRADICTIONS (ground truth) ===\n{contradictions_desc}\n\n"
        f"=== ALPHA'S PRIVATE SPEC ===\n"
        + "\n".join(f"- {s}" for s in scenario["alpha_spec"])
        + f"\n\n=== BETA'S PRIVATE SPEC ===\n"
        + "\n".join(f"- {s}" for s in scenario["beta_spec"])
        + f"\n\n=== CONVERSATION TRANSCRIPT ===\n{convo}\n\n"
        f"=== ALPHA'S FINAL DELIVERABLE ===\n{alpha_deliverable}\n\n"
        f"=== BETA'S FINAL DELIVERABLE ===\n{beta_deliverable}"
    )


def score_scenario_with_judge(conversation_lines, alpha_deliverable, beta_deliverable, scenario):
    judge_text = build_judge_text(
        conversation_lines, alpha_deliverable, beta_deliverable, scenario
    )

    dimensions = ["detection", "resolution", "cooperation", "deliverable"]
    breakdown = {}
    judge_details = []

    for dim in dimensions:
        criteria = scenario["judge_criteria"][dim]

        with kbench.chats.new(f"judge_{dim}_{scenario['name']}"):
            assessment = kbench.assertions.assess_response_with_judge(
                criteria=criteria,
                response_text=judge_text,
                judge_llm=kbench.judge_llm,
            )

        if assessment and assessment.results:
            passed = 0
            for r in assessment.results:
                judge_details.append({
                    "dim": dim,
                    "criterion": r.criterion,
                    "passed": r.passed,
                    "reason": r.reason if hasattr(r, "reason") and r.reason else "",
                })
                if r.passed:
                    passed += 1
            breakdown[dim] = (passed / len(assessment.results)) * 0.25
        else:
            breakdown[dim] = 0.0
            for c in criteria:
                judge_details.append({
                    "dim": dim,
                    "criterion": c,
                    "passed": False,
                    "reason": "Judge returned no assessment.",
                })

    total = sum(breakdown.values())
    return total, breakdown, judge_details


# --------------------------------------------------------------------------------
# 5. CONVERSATION HISTORY MANAGEMENT
# --------------------------------------------------------------------------------
MAX_HISTORY_CHARS = 12000


def truncate_history(history_lines):
    full = "\n".join(history_lines)
    if len(full) <= MAX_HISTORY_CHARS:
        return full
    while len("\n".join(history_lines)) > MAX_HISTORY_CHARS and len(history_lines) > 1:
        history_lines.pop(0)
    return "\n".join(history_lines)


# --------------------------------------------------------------------------------
# 6. THE BENCHMARK TASK
# --------------------------------------------------------------------------------
@kbench.task(name="Contradictory Instructions")
def contradiction_game(llm) -> float:
    scenario_scores = []

    for scenario in SCENARIO_SUITE:
        print(f"\n{'='*70}")
        print(f"  {scenario['name']}  [{scenario['tier']}]")
        print(f"  Task: {scenario['shared_task'][:80]}...")
        print(f"  Turn limit: {scenario['turn_limit']}")
        print(f"{'='*70}")

        alpha_system = build_player_system(scenario["shared_task"], scenario["alpha_spec"])
        beta_system = build_player_system(scenario["shared_task"], scenario["beta_spec"])

        alpha_history = []
        beta_history = []
        all_conversation = []
        alpha_last_msg = ""
        beta_last_msg = ""

        # ── CONVERSATION PHASE ──
        for turn in range(1, scenario["turn_limit"] + 1):
            print(f"\n--- Turn {turn}/{scenario['turn_limit']} ---")

            # ── Alpha's turn ──
            alpha_turn_ctx = f"--- Turn {turn} of {scenario['turn_limit']} ---\n"
            if beta_last_msg:
                alpha_turn_ctx += f"Partner's message: {beta_last_msg}\n"
            else:
                alpha_turn_ctx += "This is the start of your conversation.\n"

            alpha_history.append(alpha_turn_ctx)

            alpha_prompt = (
                alpha_system
                + "\n=== CONVERSATION SO FAR ===\n"
                + truncate_history(list(alpha_history))
                + "\n\nRespond with your MESSAGE for this turn."
            )

            with kbench.chats.new(f"alpha_{scenario['name']}_{turn}"):
                alpha_raw = llm.prompt(alpha_prompt)
            alpha_last_msg = parse_message(alpha_raw)

            alpha_history.append(f"Your message: {alpha_last_msg}")
            all_conversation.append(f"Alpha (Turn {turn}): {alpha_last_msg}")

            print(f"  ALPHA: {alpha_last_msg}")

            # ── Beta's turn ──
            beta_turn_ctx = f"--- Turn {turn} of {scenario['turn_limit']} ---\n"
            beta_turn_ctx += f"Partner's message: {alpha_last_msg}\n"

            beta_history.append(beta_turn_ctx)

            beta_prompt = (
                beta_system
                + "\n=== CONVERSATION SO FAR ===\n"
                + truncate_history(list(beta_history))
                + "\n\nRespond with your MESSAGE for this turn."
            )

            with kbench.chats.new(f"beta_{scenario['name']}_{turn}"):
                beta_raw = llm.prompt(beta_prompt)
            beta_last_msg = parse_message(beta_raw)

            beta_history.append(f"Your message: {beta_last_msg}")
            all_conversation.append(f"Beta (Turn {turn}): {beta_last_msg}")

            print(f"  BETA:  {beta_last_msg}")

        # ── EXECUTION PHASE — Final Deliverables ──
        conversation_text = "\n".join(all_conversation)

        alpha_final_prompt = build_final_prompt(
            scenario["shared_task"], scenario["alpha_spec"], conversation_text
        )
        with kbench.chats.new(f"alpha_final_{scenario['name']}"):
            alpha_final_raw = llm.prompt(alpha_final_prompt)
        alpha_deliverable = parse_deliverable(alpha_final_raw)

        beta_final_prompt = build_final_prompt(
            scenario["shared_task"], scenario["beta_spec"], conversation_text
        )
        with kbench.chats.new(f"beta_final_{scenario['name']}"):
            beta_final_raw = llm.prompt(beta_final_prompt)
        beta_deliverable = parse_deliverable(beta_final_raw)

        print(f"\n{'- '*35}")
        print(f"  FINAL DELIVERABLES")
        print(f"{'- '*35}")
        print(f"  ALPHA: {alpha_deliverable}")
        print(f"  BETA:  {beta_deliverable}")

        # ── Score via Judge LLM ──
        total, breakdown, judge_details = score_scenario_with_judge(
            all_conversation, alpha_deliverable, beta_deliverable, scenario
        )
        scenario_scores.append(total)

        # ── Print judge results ──
        print(f"\n{'- '*35}")
        print(f"  JUDGE RESULTS  |  Score: {total:.2f}/1.00")
        print(f"{'- '*35}")

        dim_labels = {
            "detection": "Detection",
            "resolution": "Resolution",
            "cooperation": "Cooperation",
            "deliverable": "Deliverable",
        }

        for dim in ["detection", "resolution", "cooperation", "deliverable"]:
            pct = int(breakdown[dim] / 0.25 * 100) if breakdown[dim] > 0 else 0
            print(f"\n  {dim_labels[dim]}  ({breakdown[dim]:.3f}/0.25 = {pct}%)")

            dim_details = [d for d in judge_details if d["dim"] == dim]
            for detail in dim_details:
                icon = "PASS" if detail["passed"] else "FAIL"
                print(f"    [{icon}] {detail['criterion']}")
                if detail["reason"]:
                    print(f"            {detail['reason']}")

    # ── Final Score ──
    final_score = sum(scenario_scores) / len(SCENARIO_SUITE)

    print(f"\n{'='*70}")
    print(f"  CONTRADICTORY INSTRUCTIONS GAME")
    print(f"  FINAL SCORE: {final_score:.4f}")
    print(f"  Per-scenario: [{', '.join(f'{s:.3f}' for s in scenario_scores)}]")
    print(f"{'='*70}")

    return final_score


# --------------------------------------------------------------------------------
# 7. EXECUTION
# --------------------------------------------------------------------------------
contradiction_game.run(kbench.llm)